# GRU Rotating Validation/Test Cross-Validation

This notebook evaluates whether the GRU can generalize to batteries it has never seen.

Each fold keeps one battery as the final test battery, one different battery as the validation battery, and trains on the remaining two batteries. This creates 12 folds from the 4 available batteries.

The scaler is fit inside each fold using only the training batteries. This prevents validation or test battery information from leaking into preprocessing.

In [ ]:
# Import the tools used for preprocessing, sequence creation, training, and evaluation.
# The notebook starts from raw data so every fold can have its own honest train-only scaler.
from pathlib import Path
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Battery_dataset.csv").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "Battery_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data file:", RAW_DATA_FILE)


In [ ]:
# Load the raw merged dataset.
# The cross-validation notebook does not reuse the current processed split because every fold needs its own scaler and windows.
raw_df = pd.read_csv(RAW_DATA_FILE)

print("Raw shape:", raw_df.shape)
print("Batteries:", sorted(raw_df["battery_id"].unique()))
raw_df.head()


In [ ]:
# Define the shared feature and target columns.
# cycle is kept for ordering only, and disT is excluded because it is constant.
FEATURE_COLUMNS = ["chI", "chV", "chT", "disI", "disV", "BCt", "SOH"]
TARGET_COLUMN = "RUL"
ID_COLUMN = "battery_id"
ORDER_COLUMN = "cycle"

WINDOW_SIZE = 10
BATCH_SIZE = 32
RANDOM_SEED = 42

INPUT_SIZE = len(FEATURE_COLUMNS)
HIDDEN_SIZE = 32
NUM_LAYERS = 1
OUTPUT_SIZE = 1
LEARNING_RATE = 0.001
MAX_EPOCHS = 500
PATIENCE = 40

model_columns = [ORDER_COLUMN, *FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]
model_df = raw_df.drop(columns=["disT"], errors="ignore")[model_columns].copy()

print("Input features:", FEATURE_COLUMNS)
print("Window size:", WINDOW_SIZE)
print("Model dataframe shape:", model_df.shape)


In [ ]:
# Define the folds before training.
# Each battery is used as a test battery three times, once with each possible validation battery.
battery_ids = ["B5", "B6", "B7", "B18"]

folds = []
fold_number = 1

for test_battery in battery_ids:
    remaining_after_test = [battery for battery in battery_ids if battery != test_battery]

    for validation_battery in remaining_after_test:
        train_batteries = [
            battery
            for battery in remaining_after_test
            if battery != validation_battery
        ]

        folds.append(
            {
                "fold": fold_number,
                "train": train_batteries,
                "validation": [validation_battery],
                "test": [test_battery],
            }
        )
        fold_number += 1

folds_df = pd.DataFrame(folds)
print("Number of folds:", len(folds))
folds_df

In [ ]:
# Scale features using training batteries only, then apply the same scaler values to validation and test.
# This repeats preprocessing inside each fold so the test battery never influences scaling.
def fit_train_min_max(train_df, feature_columns):
    feature_min = train_df[feature_columns].min()
    feature_max = train_df[feature_columns].max()
    feature_range = feature_max - feature_min
    return feature_min, feature_range


def apply_train_min_max(df, feature_columns, feature_min, feature_range):
    scaled_df = df.copy()

    for column in feature_columns:
        if feature_range[column] == 0:
            scaled_df[column] = 0.0
        else:
            scaled_df[column] = (scaled_df[column] - feature_min[column]) / feature_range[column]

    return scaled_df


In [ ]:
# Convert row-level measurements into one row per battery cycle.
# One cycle becomes one time step for the GRU.
def build_cycle_level_dataset(df):
    aggregation_rules = {column: "mean" for column in FEATURE_COLUMNS}
    aggregation_rules[TARGET_COLUMN] = "mean"

    cycle_df = (
        df[model_columns]
        .groupby([ID_COLUMN, ORDER_COLUMN], as_index=False)
        .agg(aggregation_rules)
        .sort_values([ID_COLUMN, ORDER_COLUMN])
        .reset_index(drop=True)
    )

    cycle_df[TARGET_COLUMN] = cycle_df[TARGET_COLUMN].round().astype(int)
    return cycle_df[[ORDER_COLUMN, *FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]]


In [ ]:
# Create sliding windows inside each battery only.
# A window contains feature history, and its label is the RUL at the last cycle of that window.
def create_sliding_windows(df, window_size):
    X_windows = []
    y_values = []
    metadata = []

    for battery_id, battery_df in df.groupby(ID_COLUMN):
        battery_df = battery_df.sort_values(ORDER_COLUMN).reset_index(drop=True)

        feature_values = battery_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        target_values = battery_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
        cycle_values = battery_df[ORDER_COLUMN].to_numpy()

        max_start = len(battery_df) - window_size + 1
        if max_start <= 0:
            continue

        for start_idx in range(max_start):
            end_idx = start_idx + window_size
            target_idx = end_idx - 1

            X_windows.append(feature_values[start_idx:end_idx])
            y_values.append(target_values[target_idx])
            metadata.append(
                {
                    "battery_id": battery_id,
                    "start_cycle": cycle_values[start_idx],
                    "end_cycle": cycle_values[target_idx],
                    "target_RUL": target_values[target_idx],
                }
            )

    X = np.array(X_windows, dtype=np.float32)
    y = np.array(y_values, dtype=np.float32)
    window_metadata = pd.DataFrame(metadata)
    return X, y, window_metadata


In [ ]:
# Define the GRU architecture used in the experiments.
# The optional Softplus output keeps RUL predictions non-negative while preserving smooth gradients.
class GRURULModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, output_activation=None):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, output_size)
        self.output_activation = output_activation

    def forward(self, x):
        gru_output, hidden = self.gru(x)
        final_hidden_state = hidden[-1]
        prediction = self.fc(final_hidden_state)

        if self.output_activation is not None:
            prediction = self.output_activation(prediction)

        return prediction


In [ ]:
# Build all datasets and loaders for one fold.
# This function is the core guard against leakage because scaling is fit only on fold training data.
def prepare_fold_data(fold, window_size=WINDOW_SIZE):
    train_raw = model_df[model_df[ID_COLUMN].isin(fold["train"])].copy()
    validation_raw = model_df[model_df[ID_COLUMN].isin(fold["validation"])].copy()
    test_raw = model_df[model_df[ID_COLUMN].isin(fold["test"])].copy()

    split_sets = {
        "train": set(train_raw[ID_COLUMN].unique()),
        "validation": set(validation_raw[ID_COLUMN].unique()),
        "test": set(test_raw[ID_COLUMN].unique()),
    }

    split_names = list(split_sets)
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1:]:
            overlap = split_sets[left_name] & split_sets[right_name]
            if overlap:
                raise ValueError(f"{left_name}/{right_name} overlap found: {sorted(overlap)}")

    feature_min, feature_range = fit_train_min_max(train_raw, FEATURE_COLUMNS)

    train_scaled = apply_train_min_max(train_raw, FEATURE_COLUMNS, feature_min, feature_range)
    validation_scaled = apply_train_min_max(validation_raw, FEATURE_COLUMNS, feature_min, feature_range)
    test_scaled = apply_train_min_max(test_raw, FEATURE_COLUMNS, feature_min, feature_range)

    train_cycle_df = build_cycle_level_dataset(train_scaled)
    validation_cycle_df = build_cycle_level_dataset(validation_scaled)
    test_cycle_df = build_cycle_level_dataset(test_scaled)

    X_train, y_train, train_metadata = create_sliding_windows(train_cycle_df, window_size)
    X_val, y_val, validation_metadata = create_sliding_windows(validation_cycle_df, window_size)
    X_test, y_test, test_metadata = create_sliding_windows(test_cycle_df, window_size)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    return {
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "test_dataset": test_dataset,
        "test_metadata": test_metadata,
        "shapes": {
            "X_train": X_train.shape,
            "X_val": X_val.shape,
            "X_test": X_test.shape,
        },
    }


In [ ]:
# Train one fold and return its test metrics.
# The hidden size is configurable so we can compare small GRU variants fairly.
def train_and_evaluate_fold(fold, hidden_size=HIDDEN_SIZE, window_size=WINDOW_SIZE, output_constraint="linear", max_epochs=MAX_EPOCHS, patience=PATIENCE):
    seed = RANDOM_SEED + fold["fold"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    fold_data = prepare_fold_data(fold, window_size=window_size)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        fold_data["train_dataset"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator,
    )
    val_loader = DataLoader(fold_data["val_dataset"], batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(fold_data["test_dataset"], batch_size=BATCH_SIZE, shuffle=False)

    if output_constraint == "softplus":
        output_activation = nn.Softplus()
    elif output_constraint == "linear":
        output_activation = None
    else:
        raise ValueError(f"Unsupported output_constraint: {output_constraint}")

    model = GRURULModel(
        INPUT_SIZE,
        hidden_size,
        NUM_LAYERS,
        OUTPUT_SIZE,
        output_activation=output_activation,
    )
    loss_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        total_train_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = loss_function(predictions, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * batch_X.size(0)

        average_train_loss = total_train_loss / len(train_loader.dataset)
        train_losses.append(average_train_loss)

        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                predictions = model(batch_X)
                loss = loss_function(predictions, batch_y)
                total_val_loss += loss.item() * batch_X.size(0)

        average_val_loss = total_val_loss / len(val_loader.dataset)
        val_losses.append(average_val_loss)

        if average_val_loss < best_val_loss:
            best_val_loss = average_val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_model_state)
    model.eval()

    test_predictions = []
    test_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions = model(batch_X)
            test_predictions.append(predictions)
            test_targets.append(batch_y)

    test_predictions = torch.cat(test_predictions).squeeze().numpy()
    test_targets = torch.cat(test_targets).squeeze().numpy()

    test_mae = np.mean(np.abs(test_predictions - test_targets))
    test_rmse = np.sqrt(np.mean((test_predictions - test_targets) ** 2))

    return {
        "fold": fold["fold"],
        "hidden_size": hidden_size,
        "window_size": window_size,
        "output_constraint": output_constraint,
        "train_batteries": ", ".join(fold["train"]),
        "validation_battery": ", ".join(fold["validation"]),
        "test_battery": ", ".join(fold["test"]),
        "best_epoch": best_epoch,
        "best_val_rmse": np.sqrt(best_val_loss),
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "min_test_prediction": float(np.min(test_predictions)),
        "max_test_prediction": float(np.max(test_predictions)),
        "negative_prediction_count": int(np.sum(test_predictions < 0)),
        "train_windows": fold_data["shapes"]["X_train"][0],
        "validation_windows": fold_data["shapes"]["X_val"][0],
        "test_windows": fold_data["shapes"]["X_test"][0],
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_predictions": test_predictions,
        "test_targets": test_targets,
    }


In [ ]:
# Run all rotating validation/test cross-validation folds.
# This trains 12 fresh GRU models because each test battery is paired with each possible validation battery.
fold_results = []

for fold in folds:
    print(
        f"Running fold {fold['fold']} | "
        f"validation battery: {fold['validation']} | "
        f"test battery: {fold['test']}"
    )
    result = train_and_evaluate_fold(fold)
    fold_results.append(result)
    print(
        f"Fold {result['fold']} complete | "
        f"Best Val RMSE: {result['best_val_rmse']:.2f} | "
        f"Test MAE: {result['test_mae']:.2f} | "
        f"Test RMSE: {result['test_rmse']:.2f}"
    )

In [ ]:
# Summarize fold results in one table.
# The same test battery appears in three folds, each with a different validation battery.
results_df = pd.DataFrame(
    [
        {
            "fold": result["fold"],
            "train_batteries": result["train_batteries"],
            "validation_battery": result["validation_battery"],
            "test_battery": result["test_battery"],
            "best_epoch": result["best_epoch"],
            "best_val_rmse": result["best_val_rmse"],
            "test_mae": result["test_mae"],
            "test_rmse": result["test_rmse"],
            "train_windows": result["train_windows"],
            "validation_windows": result["validation_windows"],
            "test_windows": result["test_windows"],
        }
        for result in fold_results
    ]
)

results_df

In [ ]:
# Report average performance across all rotating validation/test folds.
# This is more robust than a single fixed validation/test split.
summary_metrics = results_df[["test_mae", "test_rmse"]].agg(["mean", "std"])
summary_metrics

In [ ]:
# Plot average test RMSE by held-out battery.
# Each bar averages over the three validation choices for that test battery.
test_battery_summary_df = (
    results_df
    .groupby("test_battery", as_index=False)
    .agg(
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
    )
)

plt.figure(figsize=(8, 5))
plt.bar(test_battery_summary_df["test_battery"], test_battery_summary_df["mean_test_rmse"])
plt.errorbar(
    test_battery_summary_df["test_battery"],
    test_battery_summary_df["mean_test_rmse"],
    yerr=test_battery_summary_df["std_test_rmse"],
    fmt="none",
    color="black",
    capsize=5,
)
plt.xlabel("Held-out test battery")
plt.ylabel("Mean test RMSE (cycles)")
plt.title("GRU Rotating Validation/Test Cross-Validation RMSE")
plt.grid(axis="y")
plt.show()

test_battery_summary_df

### Cross-Validation Interpretation

This rotating validation/test protocol is stricter than the earlier fixed-validation setup.

Each battery is used as the held-out test battery three times, and each test case is evaluated with different validation choices. This matters because the validation battery affects early stopping and model selection.

If performance changes a lot across validation choices for the same test battery, the model-selection process is unstable. If one test battery remains difficult across all validation choices, the GRU is struggling to generalize to that battery's degradation behavior.

## Small GRU Hidden-Size Comparison Before XAI Candidate Selection

Before choosing a GRU model for possible XAI, we compare a small set of hidden sizes.

Only one hyperparameter changes here: `hidden_size`. The data splits, window size, features, optimizer, loss, and early-stopping rule stay the same. This keeps the experiment controlled and avoids tuning too many things on a tiny dataset.

In [ ]:
# Compare a few GRU hidden sizes using the same rotating validation/test folds.
# This can take several minutes because it trains 3 x 12 fresh GRU models.
hidden_size_candidates = [16, 32, 64]

tuning_results = []

for hidden_size in hidden_size_candidates:
    print(f"Testing hidden_size={hidden_size}")

    for fold in folds:
        result = train_and_evaluate_fold(fold, hidden_size=hidden_size)
        tuning_results.append(result)

        print(
            f"  Fold {result['fold']} | "
            f"Test {result['test_battery']} | "
            f"Best Val RMSE: {result['best_val_rmse']:.2f} | "
            f"Test RMSE: {result['test_rmse']:.2f}"
        )

In [ ]:
# Summarize the hidden-size comparison.
# The best GRU candidate is the hidden size with the lowest mean test RMSE across folds.
tuning_results_df = pd.DataFrame(
    [
        {
            "hidden_size": result["hidden_size"],
            "fold": result["fold"],
            "test_battery": result["test_battery"],
            "best_epoch": result["best_epoch"],
            "best_val_rmse": result["best_val_rmse"],
            "test_mae": result["test_mae"],
            "test_rmse": result["test_rmse"],
        }
        for result in tuning_results
    ]
)

tuning_summary_df = (
    tuning_results_df
    .groupby("hidden_size", as_index=False)
    .agg(
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
    )
)

tuning_summary_df

In [ ]:
# Choose the best hidden size by average test RMSE.
# This is the GRU candidate we would compare against the GLU candidate before deciding which model goes to XAI.
best_hidden_size_row = tuning_summary_df.loc[tuning_summary_df["mean_test_rmse"].idxmin()]
best_hidden_size = int(best_hidden_size_row["hidden_size"])

print("Best GRU hidden size:", best_hidden_size)
print(f"Mean test MAE: {best_hidden_size_row['mean_test_mae']:.2f} cycles")
print(f"Mean test RMSE: {best_hidden_size_row['mean_test_rmse']:.2f} cycles")

In [ ]:
# Plot average test RMSE for each hidden size.
# Error bars show how much performance changes across held-out battery/validation combinations.
plt.figure(figsize=(8, 5))
plt.errorbar(
    tuning_summary_df["hidden_size"],
    tuning_summary_df["mean_test_rmse"],
    yerr=tuning_summary_df["std_test_rmse"],
    fmt="o-",
    capsize=5,
)
plt.xlabel("GRU hidden size")
plt.ylabel("Mean test RMSE (cycles)")
plt.title("GRU Hidden-Size Comparison")
plt.grid(True)
plt.show()

### Choosing the GRU Candidate for XAI

The GRU candidate for XAI should not be chosen from one split only. It should be chosen using the cross-validation summary, then compared with the GLU model under the same preprocessing, windows, folds, and metrics.

If GRU is selected for XAI, the final saved model should use the chosen hidden size and the final agreed training setup.

## Small Window-Size Comparison Before Final GRU Selection

After hidden-size comparison, `hidden_size=16` is the best GRU candidate so far.

Now we test whether the length of the input sequence matters. A smaller window gives the model less history but may be easier to learn from a small dataset. A larger window gives more history but can increase complexity and reduce the number of windows.

Only `window_size` changes in this section. The hidden size stays fixed at 16.

In [ ]:
# Compare a few window sizes using the best hidden size from the previous experiment.
# This keeps the architecture mostly fixed and tests how much recent cycle history the GRU needs.
window_size_candidates = [5, 10, 20]
selected_hidden_size = 16

window_tuning_results = []

for window_size in window_size_candidates:
    print(f"Testing window_size={window_size} with hidden_size={selected_hidden_size}")

    for fold in folds:
        result = train_and_evaluate_fold(
            fold,
            hidden_size=selected_hidden_size,
            window_size=window_size,
        )
        window_tuning_results.append(result)

        print(
            f"  Fold {result['fold']} | "
            f"Test {result['test_battery']} | "
            f"Train windows: {result['train_windows']} | "
            f"Test RMSE: {result['test_rmse']:.2f}"
        )

In [ ]:
# Summarize the window-size comparison.
# The best window size is the one with the lowest mean test RMSE across held-out battery/validation combinations.
window_tuning_results_df = pd.DataFrame(
    [
        {
            "window_size": result["window_size"],
            "hidden_size": result["hidden_size"],
            "fold": result["fold"],
            "test_battery": result["test_battery"],
            "best_epoch": result["best_epoch"],
            "best_val_rmse": result["best_val_rmse"],
            "test_mae": result["test_mae"],
            "test_rmse": result["test_rmse"],
            "train_windows": result["train_windows"],
            "validation_windows": result["validation_windows"],
            "test_windows": result["test_windows"],
        }
        for result in window_tuning_results
    ]
)

window_tuning_summary_df = (
    window_tuning_results_df
    .groupby("window_size", as_index=False)
    .agg(
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        mean_train_windows=("train_windows", "mean"),
    )
)

window_tuning_summary_df

In [ ]:
# Choose the best window size by average test RMSE.
# This gives the current GRU candidate configuration before comparing against GLU.
best_window_size_row = window_tuning_summary_df.loc[window_tuning_summary_df["mean_test_rmse"].idxmin()]
best_window_size = int(best_window_size_row["window_size"])

print("Best GRU hidden size:", selected_hidden_size)
print("Best GRU window size:", best_window_size)
print(f"Mean test MAE: {best_window_size_row['mean_test_mae']:.2f} cycles")
print(f"Mean test RMSE: {best_window_size_row['mean_test_rmse']:.2f} cycles")

In [ ]:
# Plot average test RMSE for each window size.
# Error bars show whether the choice is stable across different held-out battery/validation combinations.
plt.figure(figsize=(8, 5))
plt.errorbar(
    window_tuning_summary_df["window_size"],
    window_tuning_summary_df["mean_test_rmse"],
    yerr=window_tuning_summary_df["std_test_rmse"],
    fmt="o-",
    capsize=5,
)
plt.xlabel("Window size (cycles)")
plt.ylabel("Mean test RMSE (cycles)")
plt.title("GRU Window-Size Comparison")
plt.grid(True)
plt.show()

### Current GRU Candidate Before GLU Comparison

The best GRU candidate should be selected from cross-validation, not from one split.

After comparing hidden size and window size, use the configuration with the lowest average test RMSE as the GRU candidate. This candidate can then be compared with the GLU model under the same preprocessing, folds, and metrics before deciding which model should be used for XAI.

## Non-Negative RUL Output Constraint

The baseline GRU uses a linear output layer, so it can predict negative RUL values. Negative RUL is physically invalid.

This section compares the linear-output GRU with a Softplus-output GRU. Softplus keeps predictions positive while remaining smoother than ReLU, which makes it a better first constraint for regression.

Only the output constraint changes here. Hidden size and window size use the current GRU candidate settings.

In [ ]:
# Compare linear output against Softplus output using the current best GRU settings.
# Softplus should remove negative predictions; the question is whether it also preserves or improves test error.
selected_hidden_size = 16
selected_window_size = 10
output_constraint_candidates = ["linear", "softplus"]

constraint_results = []

for output_constraint in output_constraint_candidates:
    print(f"Testing output_constraint={output_constraint}")

    for fold in folds:
        result = train_and_evaluate_fold(
            fold,
            hidden_size=selected_hidden_size,
            window_size=selected_window_size,
            output_constraint=output_constraint,
        )
        constraint_results.append(result)

        print(
            f"  Fold {result['fold']} | "
            f"Test {result['test_battery']} | "
            f"Test RMSE: {result['test_rmse']:.2f} | "
            f"Negative predictions: {result['negative_prediction_count']}"
        )

In [ ]:
# Summarize output-constraint results.
# A useful final model should balance low error with physically valid non-negative predictions.
constraint_results_df = pd.DataFrame(
    [
        {
            "output_constraint": result["output_constraint"],
            "fold": result["fold"],
            "test_battery": result["test_battery"],
            "test_mae": result["test_mae"],
            "test_rmse": result["test_rmse"],
            "min_test_prediction": result["min_test_prediction"],
            "negative_prediction_count": result["negative_prediction_count"],
        }
        for result in constraint_results
    ]
)

constraint_summary_df = (
    constraint_results_df
    .groupby("output_constraint", as_index=False)
    .agg(
        mean_test_mae=("test_mae", "mean"),
        std_test_mae=("test_mae", "std"),
        mean_test_rmse=("test_rmse", "mean"),
        std_test_rmse=("test_rmse", "std"),
        total_negative_predictions=("negative_prediction_count", "sum"),
        worst_min_prediction=("min_test_prediction", "min"),
    )
)

constraint_summary_df

In [ ]:
# Choose the output style for the GRU candidate.
# Prefer Softplus if it removes negative predictions without making cross-validation error clearly worse.
constraint_summary_df.sort_values("mean_test_rmse")

### Output Constraint Interpretation

If Softplus has similar error to the linear output and removes negative predictions, it is the better final GRU candidate because it respects the physical meaning of RUL.

If Softplus performs much worse, keep the linear model for reporting but note that raw predictions may need post-processing or a better constrained training strategy.

## Fair GRU Comparison Using the Exact GLU Folds

The previous GRU cross-validation used all rotating validation/test combinations, which is useful for robustness. For a fair GRU-vs-GLU comparison, we also run GRU on the exact same four folds reported by the GLU experiment. This keeps the train, validation, and test batteries identical, so the comparison is about the model architecture instead of the split.


In [ ]:
# Define the exact four folds used in the GLU experiment.
# Reusing the same folds makes the comparison fair because both models see the same training batteries and are tested on the same unseen batteries.
glu_comparison_folds = [
    {"fold": 1, "train": ["B7", "B18"], "validation": ["B6"], "test": ["B5"]},
    {"fold": 2, "train": ["B5", "B7"], "validation": ["B18"], "test": ["B6"]},
    {"fold": 3, "train": ["B6", "B18"], "validation": ["B5"], "test": ["B7"]},
    {"fold": 4, "train": ["B5", "B6"], "validation": ["B7"], "test": ["B18"]},
]

glu_reported_df = pd.DataFrame(
    [
        {"fold": 1, "train_batteries": "B7, B18", "validation_battery": "B6", "test_battery": "B5", "glu_mae": 17.30, "glu_rmse": 23.11, "glu_r2": 0.747},
        {"fold": 2, "train_batteries": "B5, B7", "validation_battery": "B18", "test_battery": "B6", "glu_mae": 26.90, "glu_rmse": 31.44, "glu_r2": 0.531},
        {"fold": 3, "train_batteries": "B6, B18", "validation_battery": "B5", "test_battery": "B7", "glu_mae": 15.86, "glu_rmse": 21.05, "glu_r2": 0.790},
        {"fold": 4, "train_batteries": "B5, B6", "validation_battery": "B7", "test_battery": "B18", "glu_mae": 20.60, "glu_rmse": 23.43, "glu_r2": 0.565},
    ]
)

glu_reported_df


In [ ]:
# Run GRU on the exact same four folds.
# Keep these settings equal to the selected GRU candidate so the comparison is honest and reproducible.
GRU_COMPARISON_HIDDEN_SIZE = 16
GRU_COMPARISON_WINDOW_SIZE = 10
GRU_COMPARISON_OUTPUT_CONSTRAINT = "linear"

def calculate_r2(targets, predictions):
    residual_sum_of_squares = np.sum((targets - predictions) ** 2)
    total_sum_of_squares = np.sum((targets - np.mean(targets)) ** 2)

    if total_sum_of_squares == 0:
        return np.nan

    return 1 - (residual_sum_of_squares / total_sum_of_squares)

gru_glu_comparison_results = []

for fold in glu_comparison_folds:
    result = train_and_evaluate_fold(
        fold,
        hidden_size=GRU_COMPARISON_HIDDEN_SIZE,
        window_size=GRU_COMPARISON_WINDOW_SIZE,
        output_constraint=GRU_COMPARISON_OUTPUT_CONSTRAINT,
    )

    result["test_r2"] = calculate_r2(result["test_targets"], result["test_predictions"])
    gru_glu_comparison_results.append(result)

    print(
        f"Fold {result['fold']} | "
        f"Train {result['train_batteries']} | "
        f"Validation {result['validation_battery']} | "
        f"Test {result['test_battery']} | "
        f"GRU MAE: {result['test_mae']:.2f} | "
        f"GRU RMSE: {result['test_rmse']:.2f} | "
        f"GRU R2: {result['test_r2']:.3f}"
    )


In [ ]:
# Put the GRU four-fold results into a compact table.
# This table should be read beside the GLU table because the fold definitions are now identical.
gru_glu_comparison_df = pd.DataFrame(
    [
        {
            "fold": result["fold"],
            "train_batteries": result["train_batteries"],
            "validation_battery": result["validation_battery"],
            "test_battery": result["test_battery"],
            "gru_mae": result["test_mae"],
            "gru_rmse": result["test_rmse"],
            "gru_r2": result["test_r2"],
            "best_epoch": result["best_epoch"],
            "best_val_rmse": result["best_val_rmse"],
            "negative_predictions": result["negative_prediction_count"],
        }
        for result in gru_glu_comparison_results
    ]
)

gru_glu_comparison_df


In [ ]:
# Compare GRU and GLU fold by fold.
# Positive differences mean GRU error is higher than GLU error on that same test battery.
gru_vs_glu_df = gru_glu_comparison_df.merge(
    glu_reported_df[["fold", "glu_mae", "glu_rmse", "glu_r2"]],
    on="fold",
)

gru_vs_glu_df["mae_difference_gru_minus_glu"] = gru_vs_glu_df["gru_mae"] - gru_vs_glu_df["glu_mae"]
gru_vs_glu_df["rmse_difference_gru_minus_glu"] = gru_vs_glu_df["gru_rmse"] - gru_vs_glu_df["glu_rmse"]
gru_vs_glu_df["r2_difference_gru_minus_glu"] = gru_vs_glu_df["gru_r2"] - gru_vs_glu_df["glu_r2"]

gru_vs_glu_df[[
    "fold",
    "train_batteries",
    "validation_battery",
    "test_battery",
    "gru_mae",
    "glu_mae",
    "mae_difference_gru_minus_glu",
    "gru_rmse",
    "glu_rmse",
    "rmse_difference_gru_minus_glu",
    "gru_r2",
    "glu_r2",
    "r2_difference_gru_minus_glu",
]]


In [ ]:
# Summarize the exact-fold model comparison.
# These are the numbers to use when discussing GRU vs GLU under the same evaluation protocol.
fair_model_comparison_df = pd.DataFrame(
    [
        {
            "model": "GRU",
            "mean_mae": gru_vs_glu_df["gru_mae"].mean(),
            "std_mae": gru_vs_glu_df["gru_mae"].std(),
            "mean_rmse": gru_vs_glu_df["gru_rmse"].mean(),
            "std_rmse": gru_vs_glu_df["gru_rmse"].std(),
            "mean_r2": gru_vs_glu_df["gru_r2"].mean(),
            "std_r2": gru_vs_glu_df["gru_r2"].std(),
        },
        {
            "model": "GLU",
            "mean_mae": glu_reported_df["glu_mae"].mean(),
            "std_mae": glu_reported_df["glu_mae"].std(),
            "mean_rmse": glu_reported_df["glu_rmse"].mean(),
            "std_rmse": glu_reported_df["glu_rmse"].std(),
            "mean_r2": glu_reported_df["glu_r2"].mean(),
            "std_r2": glu_reported_df["glu_r2"].std(),
        },
    ]
)

fair_model_comparison_df


In [ ]:
# Plot the fair comparison using the same four test batteries.
# Lower RMSE is better, so this figure quickly shows which model generalizes better on each held-out battery.
x = np.arange(len(gru_vs_glu_df))
bar_width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - bar_width / 2, gru_vs_glu_df["gru_rmse"], width=bar_width, label="GRU")
plt.bar(x + bar_width / 2, gru_vs_glu_df["glu_rmse"], width=bar_width, label="GLU")
plt.xticks(x, gru_vs_glu_df["test_battery"])
plt.xlabel("Held-out test battery")
plt.ylabel("Test RMSE (cycles)")
plt.title("Fair GRU vs GLU Comparison on Identical Folds")
plt.legend()
plt.grid(axis="y", alpha=0.4)
plt.show()


### How to Present This Fair Comparison

Use this section when comparing GRU and GLU directly. The earlier 12-fold GRU result answers a different question: how stable GRU is across many validation/test choices. This four-fold section answers the fair architecture question: when GRU and GLU use the same train, validation, and test batteries, which model has lower MAE/RMSE and higher R2?

If GLU used a different window size, output constraint, or preprocessing choice, update the GRU settings in this section to match it before making the final comparison.


## Supervisor-Ready Visual Checks

These plots are meant to support the discussion, not just decorate the notebook. They show why the split matters, why `cycle` is excluded from the model input, whether validation agrees with testing, where GRU makes large errors, and how the fair GRU-vs-GLU comparison looks visually.


In [ ]:
# Build one row per battery-cycle for visualization.
# This keeps the plots aligned with the model input level, where one time step is one cycle.
cycle_profile_df = build_cycle_level_dataset(model_df)

battery_summary_df = (
    cycle_profile_df
    .sort_values([ID_COLUMN, ORDER_COLUMN])
    .groupby(ID_COLUMN)
    .agg(
        cycle_count=(ORDER_COLUMN, "count"),
        first_cycle=(ORDER_COLUMN, "min"),
        last_cycle=(ORDER_COLUMN, "max"),
        start_rul=(TARGET_COLUMN, "max"),
        end_rul=(TARGET_COLUMN, "min"),
        start_soh=("SOH", "first"),
        end_soh=("SOH", "last"),
    )
    .reset_index()
)

battery_summary_df["soh_drop"] = battery_summary_df["start_soh"] - battery_summary_df["end_soh"]
battery_summary_df


In [ ]:
# Plot RUL and SOH together because they tell different parts of the story.
# RUL is perfectly linear by definition, while SOH is an observed degradation feature.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for battery_id, group in cycle_profile_df.groupby(ID_COLUMN):
    group = group.sort_values(ORDER_COLUMN)
    axes[0].plot(group[ORDER_COLUMN], group[TARGET_COLUMN], label=battery_id, linewidth=2)
    axes[1].plot(group[ORDER_COLUMN], group["SOH"], label=battery_id, linewidth=2)

axes[0].set_title("RUL decreases linearly by construction")
axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("RUL")
axes[0].grid(alpha=0.4)

axes[1].set_title("SOH shows battery-specific degradation behavior")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("SOH")
axes[1].grid(alpha=0.4)

for axis in axes:
    axis.legend(title="Battery")

plt.tight_layout()
plt.show()


In [ ]:
# Plot every model feature over cycles to inspect whether the model has real signals besides cycle number.
# If feature behavior differs between batteries, generalizing to an unseen battery becomes harder.
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=False)
axes = axes.flatten()

for axis, feature in zip(axes, FEATURE_COLUMNS):
    for battery_id, group in cycle_profile_df.groupby(ID_COLUMN):
        group = group.sort_values(ORDER_COLUMN)
        axis.plot(group[ORDER_COLUMN], group[feature], label=battery_id, linewidth=1.6, alpha=0.85)

    axis.set_title(feature)
    axis.set_xlabel("Cycle")
    axis.grid(alpha=0.3)

axes[-1].axis("off")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Battery", loc="lower right")
fig.suptitle("Cycle-Level Feature Trends by Battery", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Check feature relationships at the same cycle-level granularity used by the model.
# Strong correlations can mean features carry overlapping degradation information.
correlation_columns = [*FEATURE_COLUMNS, TARGET_COLUMN]
correlation_matrix = cycle_profile_df[correlation_columns].corr()

fig, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)

axis.set_xticks(np.arange(len(correlation_columns)))
axis.set_yticks(np.arange(len(correlation_columns)))
axis.set_xticklabels(correlation_columns, rotation=45, ha="right")
axis.set_yticklabels(correlation_columns)

for row in range(len(correlation_columns)):
    for column in range(len(correlation_columns)):
        value = correlation_matrix.iloc[row, column]
        axis.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=8)

axis.set_title("Cycle-Level Feature and RUL Correlation")
fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize which battery plays which role in each fold.
# This makes the evaluation design easier to defend because it shows that test batteries are held out by identity.
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

role_to_value = {"unused": 0, "train": 1, "validation": 2, "test": 3}
role_colors = ["#eeeeee", "#4c78a8", "#f58518", "#e45756"]
role_labels = ["Unused", "Train", "Validation", "Test"]
battery_order = ["B5", "B6", "B7", "B18"]


def plot_fold_role_matrix(fold_list, title):
    matrix = []
    y_labels = []

    for fold in fold_list:
        row = []
        for battery_id in battery_order:
            if battery_id in fold["train"]:
                row.append(role_to_value["train"])
            elif battery_id in fold["validation"]:
                row.append(role_to_value["validation"])
            elif battery_id in fold["test"]:
                row.append(role_to_value["test"])
            else:
                row.append(role_to_value["unused"])

        matrix.append(row)
        y_labels.append(f"Fold {fold['fold']}")

    fig, axis = plt.subplots(figsize=(8, max(3, len(fold_list) * 0.45)))
    axis.imshow(matrix, cmap=ListedColormap(role_colors), vmin=0, vmax=3)
    axis.set_xticks(np.arange(len(battery_order)))
    axis.set_yticks(np.arange(len(y_labels)))
    axis.set_xticklabels(battery_order)
    axis.set_yticklabels(y_labels)
    axis.set_title(title)

    for row_index, row in enumerate(matrix):
        for column_index, value in enumerate(row):
            axis.text(column_index, row_index, role_labels[value], ha="center", va="center", color="white" if value else "black", fontsize=8)

    legend_items = [Patch(facecolor=color, label=label) for color, label in zip(role_colors, role_labels)]
    axis.legend(handles=legend_items, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_fold_role_matrix(folds, "Rotating Validation/Test Fold Roles")

if "glu_comparison_folds" in globals():
    plot_fold_role_matrix(glu_comparison_folds, "Exact GLU Folds Used for Fair GRU Comparison")
else:
    print("Run the GLU comparison fold-definition cell before plotting the exact GLU fold matrix.")


In [ ]:
# Plot all rotating-CV fold errors so we can see whether one held-out battery dominates the average.
# The labels include validation and test batteries because both affect model selection and final evaluation.
if "results_df" not in globals():
    print("Run the rotating cross-validation result cells before this plot.")
else:
    fold_plot_df = results_df.copy()
    fold_plot_df["fold_label"] = (
        "F" + fold_plot_df["fold"].astype(str)
        + " | V:" + fold_plot_df["validation_battery"]
        + " | T:" + fold_plot_df["test_battery"]
    )

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].bar(fold_plot_df["fold_label"], fold_plot_df["test_mae"], color="#4c78a8")
    axes[0].set_ylabel("Test MAE (cycles)")
    axes[0].set_title("GRU Test MAE by Fold")
    axes[0].grid(axis="y", alpha=0.4)

    axes[1].bar(fold_plot_df["fold_label"], fold_plot_df["test_rmse"], color="#e45756")
    axes[1].set_ylabel("Test RMSE (cycles)")
    axes[1].set_title("GRU Test RMSE by Fold")
    axes[1].grid(axis="y", alpha=0.4)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Compare validation RMSE with test RMSE across folds.
# If points are far from the diagonal, validation performance is not fully predicting unseen-test performance.
if "results_df" not in globals():
    print("Run the rotating cross-validation result cells before this plot.")
else:
    fig, axis = plt.subplots(figsize=(7, 6))

    for test_battery, group in results_df.groupby("test_battery"):
        axis.scatter(group["best_val_rmse"], group["test_rmse"], s=80, label=test_battery)

        for _, row in group.iterrows():
            axis.text(row["best_val_rmse"], row["test_rmse"], f"F{row['fold']}", fontsize=8, ha="left", va="bottom")

    max_axis = max(results_df["best_val_rmse"].max(), results_df["test_rmse"].max()) + 5
    axis.plot([0, max_axis], [0, max_axis], linestyle="--", color="gray", label="Validation = Test")
    axis.set_xlim(0, max_axis)
    axis.set_ylim(0, max_axis)
    axis.set_xlabel("Best Validation RMSE (cycles)")
    axis.set_ylabel("Test RMSE (cycles)")
    axis.set_title("Does Validation Error Predict Test Error?")
    axis.grid(alpha=0.4)
    axis.legend(title="Test battery")
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot true vs predicted RUL curves for the fair 4-fold GRU comparison.
# This shows whether the model captures the full degradation trend or only part of it.
if "gru_glu_comparison_results" not in globals():
    print("Run the fair GRU comparison cells before this plot.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False, sharey=True)
    axes = axes.flatten()

    for axis, result in zip(axes, gru_glu_comparison_results):
        axis.plot(result["test_targets"], label="True RUL", linewidth=2)
        axis.plot(result["test_predictions"], label="Predicted RUL", linewidth=2)
        axis.set_title(f"Fold {result['fold']} | Test {result['test_battery']}")
        axis.set_xlabel("Test window index")
        axis.set_ylabel("RUL")
        axis.grid(alpha=0.4)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2)
    fig.suptitle("GRU Predictions on the Exact GLU Test Folds", fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()


In [ ]:
# Inspect residuals on the fair folds.
# Residual = predicted RUL - true RUL; positive means the model overestimates remaining life.
if "gru_glu_comparison_results" not in globals():
    print("Run the fair GRU comparison cells before this plot.")
else:
    residual_rows = []

    for result in gru_glu_comparison_results:
        for true_rul, predicted_rul in zip(result["test_targets"], result["test_predictions"]):
            residual_rows.append(
                {
                    "fold": result["fold"],
                    "test_battery": result["test_battery"],
                    "true_rul": float(true_rul),
                    "predicted_rul": float(predicted_rul),
                    "residual": float(predicted_rul - true_rul),
                    "absolute_error": float(abs(predicted_rul - true_rul)),
                }
            )

    residual_df = pd.DataFrame(residual_rows)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for test_battery, group in residual_df.groupby("test_battery"):
        axes[0].scatter(group["true_rul"], group["residual"], alpha=0.65, label=test_battery)

    axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0].set_xlabel("True RUL")
    axes[0].set_ylabel("Residual: prediction - true")
    axes[0].set_title("Residuals by True RUL")
    axes[0].grid(alpha=0.4)
    axes[0].legend(title="Test battery")

    residual_df.boxplot(column="absolute_error", by="test_battery", ax=axes[1])
    axes[1].set_title("Absolute Error by Test Battery")
    axes[1].set_xlabel("Test battery")
    axes[1].set_ylabel("Absolute error (cycles)")
    axes[1].grid(axis="y", alpha=0.4)

    fig.suptitle("")
    plt.tight_layout()
    plt.show()


In [ ]:
# Break errors into RUL stages to see where the model struggles most.
# End-of-life errors are especially important because late-life predictions are often the most actionable.
if "residual_df" not in globals():
    print("Run the residual analysis cell before this summary.")
else:
    residual_df["rul_stage"] = pd.cut(
        residual_df["true_rul"],
        bins=[-np.inf, 30, 90, np.inf],
        labels=["End-of-life (0-30)", "Middle life (31-90)", "Early life (>90)"],
    )

    stage_error_df = (
        residual_df
        .groupby(["test_battery", "rul_stage"], observed=False)
        .agg(
            mean_absolute_error=("absolute_error", "mean"),
            rmse=("residual", lambda values: np.sqrt(np.mean(values ** 2))),
            samples=("absolute_error", "count"),
        )
        .reset_index()
    )

    display(stage_error_df)

    pivot_stage_df = stage_error_df.pivot(index="rul_stage", columns="test_battery", values="mean_absolute_error")
    pivot_stage_df.plot(kind="bar", figsize=(10, 5))
    plt.ylabel("Mean absolute error (cycles)")
    plt.xlabel("RUL stage")
    plt.title("GRU Error by Life Stage on Fair Test Folds")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize the fair GRU-vs-GLU differences directly.
# Values above zero mean GRU has larger error than GLU on the same fold.
if "gru_vs_glu_df" not in globals():
    print("Run the fair GRU-vs-GLU comparison cells before this plot.")
else:
    x = np.arange(len(gru_vs_glu_df))
    labels = "F" + gru_vs_glu_df["fold"].astype(str) + " | Test " + gru_vs_glu_df["test_battery"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(labels, gru_vs_glu_df["mae_difference_gru_minus_glu"], color="#f58518")
    axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0].set_ylabel("MAE difference (GRU - GLU)")
    axes[0].set_title("MAE Gap on Identical Folds")
    axes[0].grid(axis="y", alpha=0.4)

    axes[1].bar(labels, gru_vs_glu_df["rmse_difference_gru_minus_glu"], color="#e45756")
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set_ylabel("RMSE difference (GRU - GLU)")
    axes[1].set_title("RMSE Gap on Identical Folds")
    axes[1].grid(axis="y", alpha=0.4)

    for axis in axes:
        axis.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()


### What These Visuals Add

The first plots justify the preprocessing choices: RUL is linear by construction, but SOH and the electrical features have battery-specific degradation patterns. The fold matrix shows that evaluation is battery-wise, not row-wise. The validation-vs-test plot checks whether validation is a reliable model-selection signal. The prediction and residual plots show where GRU fails, and the GRU-vs-GLU gap plot gives a clear visual argument that GLU is currently the stronger candidate under the exact same folds.
